# 06 - Export Model

This notebook exports the trained model artifacts into the project’s model directory so that the existing Healix runtime can load them.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

# Find repository root
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
repo_root = None
for candidate in candidates:
    if (candidate / "training" / "data" / "processed" / "X_train.csv").exists():
        repo_root = candidate
        break

# Load training data and train a final model
X_train = pd.read_csv(repo_root / "training" / "data" / "processed" / "X_train.csv")
y_train = pd.read_csv(repo_root / "training" / "data" / "processed" / "y_train.csv")["target"].astype(np.int16)

# Downcast: evidence columns are strictly 0/1, AGE fits in float32.
evidence_cols = [c for c in X_train.columns if c not in ("AGE", "SEX_M")]
X_train[evidence_cols] = X_train[evidence_cols].astype(np.int8)
X_train["SEX_M"] = X_train["SEX_M"].astype(np.int8)
X_train["AGE"] = X_train["AGE"].astype(np.float32)

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",  # explicit -- benchmarked ~30min on the full 768k-row set vs. much slower with exact
    random_state=42,
    n_jobs=8,
)
model.fit(X_train, y_train)

# Export artifacts to both the training workspace and the runtime models directory.
# Single canonical filename (xgboost_model.pkl) -- a prior version of this notebook also
# dumped the same XGBoost model under "logreg_final.pkl", which mislabeled the algorithm
# for anyone reading the runtime model_loader default path. Not repeating that here.
training_models_dir = repo_root / "training" / "models"
runtime_models_dir = repo_root / "models"
for target_dir in [training_models_dir, runtime_models_dir]:
    target_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, training_models_dir / "xgboost_model.pkl")
joblib.dump(model, runtime_models_dir / "xgboost_model.pkl")

# Load label encoder and feature names from preprocessing step
label_encoder = joblib.load(repo_root / "training" / "models" / "label_encoder.joblib")
feature_names = json.loads((repo_root / "training" / "models" / "feature_names.json").read_text(encoding="utf-8"))

for target_dir in [training_models_dir, runtime_models_dir]:
    joblib.dump(label_encoder, target_dir / "label_encoder.joblib")
    (target_dir / "feature_names.json").write_text(json.dumps(feature_names, indent=2), encoding="utf-8")

print("Exported artifacts:")
for path in [training_models_dir / "xgboost_model.pkl", runtime_models_dir / "xgboost_model.pkl", runtime_models_dir / "feature_names.json", runtime_models_dir / "label_encoder.joblib"]:
    print(path)

Exported artifacts:
C:\healix-ai-medical-assistant\training\models\xgboost_model.pkl
C:\healix-ai-medical-assistant\models\xgboost_model.pkl
C:\healix-ai-medical-assistant\models\feature_names.json
C:\healix-ai-medical-assistant\models\label_encoder.joblib


## Export location

The exported artifacts are placed under the training models folder so they can be used by the Healix ML workflow during development and presentation. For runtime integration, the project’s existing loader expects files under the repository-level models directory; those can be copied there if needed.